# Fine-tune VietOCR on Vietnamese legal documents

Fine-tunes **VietOCR** (VGG19-BN + Transformer, [pbcquoc/vietocr](https://github.com/pbcquoc/vietocr)) as a second text-line **recogniser**, next to the PaddleOCR one in `vi_legal_ocr_finetune.ipynb`. It reads the **same data** (`rec/` line crops, `dict/vi_compact.txt`), so a `data.tar` / `data.zip` built for Paddle is restored and reused as it is, and the two recognisers are scored on the same held-out lines with the same metric (CER, and the share of it that is tone marks).

**Why not just `vietocr` as shipped.** Its vocabulary (229 characters) has no `§ № ₫ ‘ ’ “ ” – — …` and no Latin-1 letters (foreign names), so it cannot spell them. The vocabulary here is VietOCR's plus the missing characters *appended*, and the pretrained embedding / output rows are carried over (a new letter starts from its base letter: ế from ê). The training loop is ours (`ocr_train/vietocr_train.py`), not `vietocr.model.trainer.Trainer`: that one imports `imgaug` (broken on Colab's numpy 2), never writes a checkpoint and picks the best model by whole-line accuracy. Here a Colab disconnect costs at most `VOCR_SAVE` steps and the best model is the one with the lowest CER.

**How it runs.** Every step is `scripts/run_vietocr.sh` (the cells below call it phase by phase), no virtualenv: VietOCR needs only PyTorch, which Colab ships.

**Runtime:** Runtime → Change runtime type → GPU. Checkpoints (`latest.pt` ≈ 0.5 GB with the optimiser, `best.pth` ≈ 0.15 GB) go to `OUT_DIR` on Drive if it is mounted.

In [ ]:
#@title 0. Settings
REPO_URL = "https://github.com/xuanvinh1997/vi-legal-ocr-training.git" #@param {type:"string"}
OUT_DIR = "/content/drive/MyDrive/ocr-vi-legal" #@param {type:"string"}
DATA_ARCHIVE = "/content/drive/MyDrive/Workspaces/Shared/Datasets/OCR/data.zip" #@param {type:"string"}
#@markdown `DATA_ARCHIVE`: the `data.zip` / `data.tar` of the PaddleOCR notebook (any path). Empty or missing: `OUT_DIR/data.tar`, else the data is built from scratch (≈ an hour).
VOCR_TAG = "vietocr" #@param {type:"string"}
VOCR_EPOCHS = 3 #@param {type:"integer"}
VOCR_BATCH = 32 #@param {type:"integer"}
VOCR_HEIGHT = 32 #@param [32, 48] {type:"raw"}
#@markdown `VOCR_HEIGHT`: 32 is what the pretrained weights saw. Taller shows tone marks better but starts further from the pretrained model, and the positional encoding then holds fewer pixels of width (48: 1364 px, so the widest lines are squeezed); untested.
VOCR_LR_SCALE = 0.4 #@param {type:"number"}
VOCR_EVAL = 2000 #@param {type:"integer"}
VOCR_SAVE = 2000 #@param {type:"integer"}
VOCR_METRICS = 1500 #@param {type:"integer"}
VOCR_AMP = True #@param {type:"boolean"}
RESUME = False #@param {type:"boolean"}
#@markdown `RESUME`: continue an interrupted run of the same `VOCR_TAG` from `latest.pt` (same schedule). For a longer second round use `INIT_FROM`.
INIT_FROM = "" #@param {type:"string"}
#@markdown `INIT_FROM`: config of an earlier run (`OUT_DIR/configs/vietocr.yml`) → start from its `best.pth` into a new `VOCR_TAG`. Never overwrites the first run.
EVAL_LIST = "val_list.txt" #@param {type:"string"}
#@markdown Only used when the data has to be built:
CORPUS_DOCS = 3000 #@param {type:"integer"}
SYNTH_LINES = 300000 #@param {type:"integer"}
SYNTH_PAGES = 4000 #@param {type:"integer"}
PDF_DIR = "" #@param {type:"string"}
PDF_REPEAT = 3 #@param {type:"integer"}

In [ ]:
import json, pathlib, random, shlex, subprocess

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("not on Colab")
if not pathlib.Path("/content/drive").exists():
    OUT_DIR = "/content/out"
    print("Drive not mounted: checkpoints stay on this VM in", OUT_DIR)

DATA, CODE = pathlib.Path("/content/data"), pathlib.Path("/content/ocr")
if (CODE / ".git").exists():
    !git -C {CODE} pull -q
else:
    !git clone -q --depth 1 {REPO_URL} {CODE}

SCRIPT = f"{CODE}/scripts/run_vietocr.sh"
ENV = " ".join(f"{k}={shlex.quote(str(v))}" for k, v in dict(
    CODE=CODE, DATA=DATA, OUT=OUT_DIR, VOCR_TAG=VOCR_TAG, VOCR_EPOCHS=VOCR_EPOCHS, VOCR_BATCH=VOCR_BATCH, VOCR_HEIGHT=VOCR_HEIGHT,
    VOCR_LR_SCALE=VOCR_LR_SCALE, VOCR_EVAL=VOCR_EVAL, VOCR_SAVE=VOCR_SAVE, VOCR_METRICS=VOCR_METRICS, VOCR_AMP=int(VOCR_AMP),
    RESUME=int(RESUME), INIT_FROM=INIT_FROM, EVAL_LIST=EVAL_LIST, CORPUS_DOCS=CORPUS_DOCS, SYNTH_LINES=SYNTH_LINES,
    SYNTH_PAGES=SYNTH_PAGES, PDF_DIR=PDF_DIR, PDF_REPEAT=PDF_REPEAT).items())
print(ENV)
!nvidia-smi -L

## 1. Install

PyTorch comes with Colab. `vietocr` is installed with `--no-deps` (its dependencies pin `imgaug`, which does not import on numpy 2; the part of VietOCR that needs it is not used) plus `einops` and `gdown`. The repository is installed in editable mode.

In [ ]:
!{ENV} PHASES="setup" bash {SCRIPT}

## 2. Data

Restored from `DATA_ARCHIVE` (or `OUT_DIR/data.tar`) if the Paddle run left one, otherwise built exactly as in the PaddleOCR notebook: real legal text from the CC-BY-4.0 corpus [th1nhng0/vietnamese-legal-documents](https://huggingface.co/datasets/th1nhng0/vietnamese-legal-documents) set in Times-style fonts with scan degradations, plus synthetic A4 pages and, if `PDF_DIR` is set, your own born-digital PDFs.

In [ ]:
if (DATA / "rec/train_list.txt").exists():
    print("data already on this VM")
elif DATA_ARCHIVE and pathlib.Path(DATA_ARCHIVE).exists():
    print("restoring", DATA_ARCHIVE)
    if DATA_ARCHIVE.endswith(".zip"):
        !unzip -q -o {shlex.quote(DATA_ARCHIVE)} -d /content
    else:
        !tar -C /content -xf {shlex.quote(DATA_ARCHIVE)}
elif pathlib.Path(OUT_DIR, "data.tar").exists():
    print("restoring", OUT_DIR + "/data.tar")
    !tar -C /content -xf {OUT_DIR}/data.tar
else:
    print("no archive found: building the dataset")
!{ENV} PHASES="data" bash {SCRIPT}

In [ ]:
# Look at the data before spending GPU hours on it.
import matplotlib.pyplot as plt
from PIL import Image

rec = [l.split("\t")[:2] for l in (DATA / "rec/train_list.txt").read_text(encoding="utf-8").splitlines()]
fig, axes = plt.subplots(8, 1, figsize=(12, 7))
for ax, (rel, text) in zip(axes, random.sample(rec, 8)):
    ax.imshow(Image.open(DATA / "rec" / rel)); ax.set_title(text[:110], fontsize=8, loc="left"); ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Config and baseline

`config` writes `OUT_DIR/configs/<VOCR_TAG>.yml`: the vocabulary (VietOCR's + what our dictionary adds), the image width (the widest line of the data, so no line is squeezed; the positional encoding limits it to 2048 px at height 32) and the schedule. `baseline` scores VietOCR's own `vgg_transformer.pth` on the held-out lines: that is the number to beat, and `tone-error-share` is how much of its error is tone marks.

In [ ]:
!{ENV} PHASES="config baseline" bash {SCRIPT}

## 4. Train

Width-bucketed batches (VietOCR has no padding mask on the encoder, so a batch is one width), AdamW + one-cycle schedule, label smoothing, 5 % masked input characters (VietOCR's own recipe), mixed precision on the GPU (bfloat16 where available, float16 with loss scaling on a T4). No image augmentation: the lines are already degraded like scans. Every `VOCR_EVAL` steps the same `VOCR_METRICS` held-out lines are decoded; `best.pth` is the lowest CER so far, `latest.pt` the full state for `RESUME`.

After a disconnect: set `RESUME = True` in the settings, run cells 0 and 1 again, then this cell.

In [ ]:
!{ENV} PHASES="train" bash {SCRIPT}

## 5. Evaluate and export

`eval` scores the fine-tuned `best.pth` on the same lines as the baseline. `export` writes `OUT_DIR/export/<VOCR_TAG>/{weights.pth,config.yml}`, which load into the stock `vietocr` `Predictor` (below).

In [ ]:
!{ENV} PHASES="eval export" bash {SCRIPT}

In [ ]:
# The exported model through VietOCR's own Predictor, on a few held-out lines.
import torch
from PIL import Image
from vietocr.tool.config import Cfg
from vietocr.tool.predictor import Predictor

export_dir = pathlib.Path(OUT_DIR) / "export" / VOCR_TAG
config = Cfg.load_config_from_file(export_dir / "config.yml")
config["weights"], config["device"] = str(export_dir / "weights.pth"), "cuda:0" if torch.cuda.is_available() else "cpu"
predictor = Predictor(config)
val = [l.split("\t")[:2] for l in (DATA / "rec" / EVAL_LIST).read_text(encoding="utf-8").splitlines() if l]
for rel, text in random.sample(val, 8):
    print("ref :", text); print("pred:", predictor.predict(Image.open(DATA / "rec" / rel))); print()

## Using the result

* VietOCR: `Predictor(config)` as above; `config.yml` carries the extended `vocab`, `weights.pth` the matching weights. Give `weights` as an absolute path.
* It recognises one text line at a time. Lines come from a detector (the PaddleOCR detection model of the other notebook, or any other).
* **Second round** (more data, more epochs): set `INIT_FROM` to `OUT_DIR/configs/<VOCR_TAG>.yml` and a new `VOCR_TAG` (e.g. `vietocr_r2`), run `config`, `train`, `eval`, `export` again. The first run is never touched.
* **Compare with PaddleOCR:** the `eval` and `baseline` lines print the same report as `eval-rec` in the other notebook (CER, CER ignoring tones, tone-error share) on `val_list.txt`.